# File 3b - Cohort Night-HR Baseline (BIDSleep) - *verification layer*
### `03b_cohort_baseline.ipynb`

Answers the professor's **"n=1"** critique by placing the personal circadian
baseline (File 3 Part A) inside a **47-person cohort measured with the same
sensor** - the Apple Watch.

**Dataset:** [BIDSleep](https://physionet.org/content/bidsleep-dataset/1.0.0/) -
47 healthy adults, 253 nights, instantaneous Apple-Watch HR (~0.2 Hz) +
accelerometry + EEG sleep-stage labels. **Open Data Commons Attribution License.**

**Why this dataset (and its honest limits):**
- **Same device** as the subject's own data -> Apple-Watch-vs-Apple-Watch, no
  cross-sensor confound (unlike a chest-strap cohort).
- Real clock timestamps (stored in US Eastern) -> we can pool by **hour-of-day**.
- **Night only.** BIDSleep records during sleep, so the cohort baseline is an
  **empirical night-HR band by clock hour**, NOT a 24h cosinor. We do **not**
  extrapolate a daytime curve from night data - that would be dishonest.
- The overnight trough (00:00-05:00) is exactly where the personal record is
  sparsest and where "n=1" is most vulnerable, so this is the highest-value place
  to show cohort agreement.

**Guardrails:**
- Cohort baseline is for **context/verification only** - it never relabels the
  subject's anomalies (no circularity).
- `MixedLM` random effect **by subject** (guide's File 3b spec) reports how much
  HR varies between people vs within the night.
- Circadian comparison is **local-clock-hour vs local-clock-hour** (Adelaide for
  the subject, US-Eastern for the cohort) - the correct alignment, since the
  rhythm tracks each person's own day.

## 1. Get the data (HR files only)

BIDSleep is 27.9 GB uncompressed, but the accelerometry (`motion.csv`) is the
bulk. We only need `hr.csv` (+ optional `labels.mat`), which is tiny. Download
just those with `wget` (open dataset - accept the license on the page first):

```bash
wget -r -N -c -np -nH --cut-dirs=3 -A "hr.csv,labels.mat" \
     https://physionet.org/files/bidsleep-dataset/1.0.0/
```

Then point `BIDSLEEP_DIR` at the folder that contains the `Bidslab00`, `Bidslab01`, ...
subject folders. `TODO(user-input)`: set this path.

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# TODO(user-input): folder containing Bidslab00/, Bidslab01/, ... after download.
BIDSLEEP_DIR = r"C:\Project\Apple Health Data\data\bidsleep"
COHORT_TZ    = "US/Eastern"     # BIDSleep start times are stored in US Eastern
PROC_DIR     = r"C:\Project\Apple Health Data\data\processed"

subject_dirs = sorted(glob.glob(os.path.join(BIDSLEEP_DIR, "Bidslab*")))
HAVE_DATA = len(subject_dirs) > 0
print(f"BIDSLEEP_DIR = {BIDSLEEP_DIR}")
print(f"subject folders found: {len(subject_dirs)}"
      + ("" if HAVE_DATA else "  <-- none yet; run the wget above, then set BIDSLEEP_DIR."))

## 2. Load every night's `hr.csv` into one cohort table

For each night we read `hr.csv` (Unix-time timestamp + HR in bpm), convert to the
cohort's **local clock time** (US Eastern), and record `subject_id` + `clock_hour`.
Timestamps are handled robustly: absolute Unix if large, else treated as an offset
from `recStart` (read from `labels.mat`).

In [ ]:
from scipy.io import loadmat

def _read_recstart(night_dir):
    """recStart (Unix seconds) from labels.mat, if available."""
    p = os.path.join(night_dir, "labels.mat")
    if not os.path.exists(p):
        return None
    try:
        m = loadmat(p)
        rs = np.array(m.get("recStart")).ravel()
        return float(rs[0]) if rs.size else None
    except Exception:
        return None

def load_night(night_dir, subject_id):
    hp = os.path.join(night_dir, "hr.csv")
    if not os.path.exists(hp):
        return None
    df = pd.read_csv(hp, header=None, names=["ts", "hr"])
    df = df.apply(pd.to_numeric, errors="coerce").dropna()
    if df.empty:
        return None
    ts = df["ts"].to_numpy(dtype=float)
    if np.median(ts) < 1e6:                 # relative offsets -> add recStart
        rs = _read_recstart(night_dir)
        if rs is None:
            return None
        ts = ts + rs
    t = (pd.to_datetime(ts, unit="s", utc=True).tz_convert(COHORT_TZ))
    out = pd.DataFrame({"subject_id": subject_id, "datetime": t,
                        "hr": df["hr"].to_numpy(dtype=float)})
    out = out[(out["hr"] >= 30) & (out["hr"] <= 220)]   # physiological sanity
    out["clock_hour"] = out["datetime"].dt.hour
    out["night_id"] = os.path.basename(night_dir)
    return out

parts = []
if HAVE_DATA:
    for sd in subject_dirs:
        sid = os.path.basename(sd)
        nights = [d for d in glob.glob(os.path.join(sd, "*")) if os.path.isdir(d)]
        # some layouts put hr.csv directly under the subject folder
        if not nights and os.path.exists(os.path.join(sd, "hr.csv")):
            nights = [sd]
        for nd in nights:
            r = load_night(nd, sid)
            if r is not None:
                parts.append(r)

cohort = (pd.concat(parts, ignore_index=True) if parts
          else pd.DataFrame(columns=["subject_id", "datetime", "hr", "clock_hour", "night_id"]))
print(f"cohort HR readings: {len(cohort):,} | subjects: {cohort['subject_id'].nunique()} "
      f"| nights: {cohort['night_id'].nunique() if len(cohort) else 0}")
if len(cohort):
    print("readings per clock-hour:\n", cohort["clock_hour"].value_counts().sort_index())

## 3. Cohort night-HR baseline

Two complementary views of "normal night HR for this cohort":

1. **Empirical band by clock hour** - per-subject mean HR at each night hour, then
   the cohort **mean** and **2.5-97.5 percentile band across subjects**. This is
   the honest "cloud of individual night curves" the personal curve must fall into.
2. **`MixedLM` (guide's File 3b spec)** - `hr ~ C(clock_hour)` with a **random
   intercept per subject**, reporting the between-subject SD, within-night
   residual SD, and the **ICC** (how much HR variance is between people).

In [ ]:
NIGHT_HOURS = [21, 22, 23, 0, 1, 2, 3, 4, 5, 6, 7, 8]   # clock hours BIDSleep covers

cohort_curve = pd.DataFrame(index=NIGHT_HOURS)
mixed_summary = None
if len(cohort):
    night = cohort[cohort["clock_hour"].isin(NIGHT_HOURS)].copy()

    # per-subject mean HR at each clock hour
    subj_hour = (night.groupby(["subject_id", "clock_hour"])["hr"]
                 .mean().reset_index(name="hr"))

    # (1) empirical cohort band across subjects
    g = subj_hour.groupby("clock_hour")["hr"]
    cohort_curve = pd.DataFrame({
        "cohort_mean": g.mean(),
        "p2_5":  g.quantile(0.025),
        "p97_5": g.quantile(0.975),
        "subj_sd": g.std(),
        "n_subj": g.size(),
    }).reindex(NIGHT_HOURS)
    print(cohort_curve.round(1).to_string())

    # (2) MixedLM random intercept by subject
    try:
        import statsmodels.formula.api as smf
        sub = subj_hour.copy()
        sub["clock_hour"] = sub["clock_hour"].astype("category")
        mdf = smf.mixedlm("hr ~ C(clock_hour)", data=sub, groups=sub["subject_id"]).fit()
        gv = float(mdf.cov_re.iloc[0, 0]); rv = float(mdf.scale)
        icc = gv / (gv + rv)
        mixed_summary = {"between_subject_var": gv, "residual_var": rv, "ICC": icc}
        print(f"\nMixedLM  between-subject SD={gv**0.5:.1f} bpm  "
              f"residual SD={rv**0.5:.1f} bpm  ICC={icc:.2f}")
    except ImportError:
        print("\nstatsmodels not installed - empirical band still available; "
              "`pip install statsmodels` to add the MixedLM.")
else:
    print("no cohort data loaded - set BIDSLEEP_DIR in cell 1 and re-run.")

In [ ]:
import matplotlib.pyplot as plt
if len(cohort_curve.dropna(how="all")):
    order = NIGHT_HOURS
    x = np.arange(len(order))
    cc = cohort_curve.reindex(order)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.fill_between(x, cc["p2_5"], cc["p97_5"], alpha=0.25, color="teal",
                    label="cohort 2.5-97.5% (across 47 subjects)")
    ax.plot(x, cc["cohort_mean"], "o-", color="teal", label="cohort mean night HR")
    ax.set_xticks(x); ax.set_xticklabels(order)
    ax.set(xlabel="clock hour (local)", ylabel="HR (bpm)",
           title="BIDSleep cohort night-HR baseline (Apple Watch, n=47)")
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print("(no plot - load data first)")

## 4. Verify the personal baseline against the cohort

Load the personal Standard Definitions (File 3 Part A) and take the personal
**expected HR by clock hour** over the same night hours. Then ask, per hour:
**does the personal circadian curve fall inside the cohort's 2.5-97.5% band?**

This is the direct rebuttal to "n=1": if the subject's own overnight HR curve sits
within the distribution of 47 other Apple-Watch wearers, the personal baseline is
shown to be *typical*, not an artefact of a single noisy recording.

In [ ]:
STD = os.path.join(PROC_DIR, "standard_definitions.parquet")
verify = None
if os.path.exists(STD) and len(cohort_curve.dropna(how="all")):
    std = pd.read_parquet(STD)
    if std["datetime"].dt.tz is None:
        std["datetime"] = std["datetime"].dt.tz_localize("Australia/Adelaide")
    std["clock_hour"] = std["datetime"].dt.hour

    # personal expected-HR curve by clock hour (cosinor prediction, REST-based)
    personal = (std.groupby("clock_hour")["expected_hr"].mean()
                .reindex(NIGHT_HOURS).rename("personal_expected"))

    verify = cohort_curve.join(personal)
    verify["inside_band"] = ((verify["personal_expected"] >= verify["p2_5"]) &
                             (verify["personal_expected"] <= verify["p97_5"]))
    verify["z_vs_cohort"] = ((verify["personal_expected"] - verify["cohort_mean"])
                             / verify["subj_sd"])
    inside_frac = verify["inside_band"].mean()
    print(verify[["personal_expected", "cohort_mean", "p2_5", "p97_5",
                  "inside_band", "z_vs_cohort"]].round(2).to_string())
    print(f"\npersonal night curve inside cohort band: {inside_frac:.0%} of night hours")
    print(f"median |z| vs cohort: {verify['z_vs_cohort'].abs().median():.2f}")
elif not os.path.exists(STD):
    print("standard_definitions.parquet missing - run File 3 Part A first.")
else:
    print("no cohort data - set BIDSLEEP_DIR and re-run.")

In [ ]:
if verify is not None:
    order = NIGHT_HOURS; x = np.arange(len(order)); v = verify.reindex(order)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.fill_between(x, v["p2_5"], v["p97_5"], alpha=0.25, color="teal",
                    label="cohort 2.5-97.5% (n=47 Apple Watch)")
    ax.plot(x, v["cohort_mean"], "o-", color="teal", alpha=0.7, label="cohort mean")
    ax.plot(x, v["personal_expected"], "s-", color="crimson", lw=2,
            label="THIS subject (personal cosinor)")
    ax.set_xticks(x); ax.set_xticklabels(order)
    ax.set(xlabel="clock hour (local)", ylabel="HR (bpm)",
           title="Personal night HR vs 47-person Apple-Watch cohort")
    ax.legend(); plt.tight_layout(); plt.show()

    os.makedirs(PROC_DIR, exist_ok=True)
    verify.to_parquet(os.path.join(PROC_DIR, "cohort_verification.parquet"))
    print("saved cohort_verification.parquet")

## File 3b output summary

Written to `data/processed/`:
- `cohort_verification.parquet` - per-hour: personal expected HR, cohort mean /
  band, `inside_band`, `z_vs_cohort`.

**How to report it (professor-facing):**
> "The overnight HR trough of this subject falls within the 2.5-97.5% band of a
> 47-person Apple-Watch cohort (BIDSleep) at *X%* of night hours, median |z| = *Y*.
> The personal circadian baseline is therefore consistent with an independent
> same-device population, addressing the single-subject concern for the night
> window where the personal record is sparsest."

**Honest caveats to state alongside it:**
- BIDSleep is **night-only** - this verifies the overnight window, not the full day.
- Cohort HR is Apple-Watch PPG (same modality as the subject); 47 healthy adults,
  US-based; local-clock-hour alignment assumes comparable habitual sleep timing.
- The cohort is used **only** to contextualise the personal baseline - it is never
  fed back as a label into the GNN or the anomaly detectors (no circularity).